# Fridge Chef: Solutions Notebook

This notebook contains complete solutions for all three phases:
- **Phase 1**: Rule-based think function (working)
- **Phase 2**: LLM-powered think function (working)
- **Phase 3**: ML classifier think function (skeleton)

In [ ]:
# Setup
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists('coding-exercises'):
        !cd coding-exercises && git pull
    else:
        !git clone -b week3 https://github.com/eth-bmai-fs26/coding-exercises.git
    os.chdir('coding-exercises/fridge_chef')

sys.path.insert(0, '.')

!pip install openai --quiet
print('Setup complete!')

In [1]:
from fridge_chef import *
from fridge_chef.game import KitchenWorld
from fridge_chef.data import ChefState
from fridge_chef.agent import TOOLS_DESCRIPTION, parse_action
print('Fridge Chef loaded!')

# Preview the kitchen
chef, world, tools = create_game()
print()
print(world.fridge_summary())
print()
print('Available tools:')
print(TOOLS_DESCRIPTION)

Fridge Chef loaded!

Fridge (12 items): eggs, milk, butter, cheddar cheese, bread, tomatoes, onions, garlic, bell pepper, chicken breast, rice, soy sauce

Available tools:
Available tools (use exactly one per turn):

ACTION: check_fridge()
  See what ingredients are currently in your fridge.

ACTION: search_recipes(ingredient="eggs")
  Search for recipes that use a specific ingredient from your fridge.

ACTION: get_ingredients(recipe="omelette")
  Get the full ingredient list for a recipe. This also sets it as your
  chosen recipe. Compare against your fridge to see what's missing.

ACTION: add_to_shopping_list(item="paprika")
  Add a missing ingredient to your shopping list. Only add items you
  don't already have in the fridge.

ACTION: done()
  Signal that you're finished — recipe chosen and shopping list ready.
  Make sure you've chosen a recipe with get_ingredients() first!


---
## Play the Game Yourself!

In [ ]:
from fridge_chef.interactive import play_interactive

game = play_interactive()

---
## Solution: Phase 1 — Rule-Based Think Function

Strategy: follow a linear pipeline to fulfill the user's request:

1. **Check fridge** — see what we have
2. **Extract ingredient** — scan the request for a fridge item by keyword
3. **Search recipes** — find recipes using that ingredient
4. **Pick recipe** — choose the first match
5. **Get ingredients** — see what the recipe needs
6. **Add missing items** — add each one to the shopping list
7. **Done** — signal completion

**The fundamental limitation**: step 2 only works when the user names an ingredient directly
(`"I want to cook eggs"`). For natural requests like `"something comforting for a cold evening"`,
no keyword matches — so the agent blindly falls back to the first fridge item.
The pipeline still *completes*, but the recipe choice is meaningless.

In [5]:
def _extract_ingredient(user_request: str, fridge: list[str]) -> str:
    """Try to find a fridge item mentioned in the request.

    Works for: "I want to cook something with eggs" → "eggs"
    Fails for: "something comforting for a cold evening" → falls back to fridge[0]

    This is the hard ceiling of the rule-based approach — interpreting intent
    requires language understanding, not string matching.
    """
    request_lower = user_request.lower()
    for item in fridge:
        if item.lower() in request_lower:
            return item
    # No match — fall back to first fridge item regardless of what the user wants
    return fridge[0] if fridge else "eggs"


def think_rule_based(chef: ChefState, world: KitchenWorld, history: list[dict], user_request: str) -> str:
    """Rule-based kitchen agent following a linear pipeline."""

    # --- Step 1: Check fridge if we haven't yet ---
    if not chef.fridge_contents:
        return 'ACTION: check_fridge()'

    # --- Step 2: Search recipes if we haven't yet ---
    if not chef.possible_recipes:
        ingredient = _extract_ingredient(user_request, chef.fridge_contents)
        return f'ACTION: search_recipes(ingredient="{ingredient}")'

    # --- Step 3: Pick a recipe and get its ingredients ---
    if not chef.chosen_recipe:
        recipe = chef.possible_recipes[0]
        return f'ACTION: get_ingredients(recipe="{recipe}")'

    # --- Step 4: Add missing ingredients to shopping list one by one ---
    if chef.needed_ingredients:
        fridge_lower = [item.lower() for item in chef.fridge_contents]
        shopping_lower = [item.lower() for item in chef.shopping_list]
        for ingredient in chef.needed_ingredients:
            if ingredient.lower() not in fridge_lower and ingredient.lower() not in shopping_lower:
                return f'ACTION: add_to_shopping_list(item="{ingredient}")'

    # --- Step 5: All done! ---
    return 'ACTION: done()'

In [6]:
USER_REQUEST = "I want something comforting for a cold evening"

result_rb = play_rule_based(think_rule_based, user_request=USER_REQUEST, max_turns=20, delay=0.05)
print(f"\nRequest:  '{USER_REQUEST}'")
print(f"Result:   {'Complete!' if result_rb['completed'] else 'Timed out'}")
print(f"Recipe:   {result_rb['recipe']}  ← picked because it was first in the list, not because it's comforting")
print(f"Shopping: {', '.join(result_rb['shopping_list']) or 'nothing needed'}")
print(f"Turns:    {result_rb['turns']}")


Request:  'I want something comforting for a cold evening'
Result:   Complete!
Recipe:   omelette  ← picked because it was first in the list, not because it's comforting
Shopping: salt, pepper
Turns:    7


---
## Solution: Phase 2 — LLM-Powered Think Function

The rule-based agent broke on `"I want something comforting for a cold evening"` — it ignored
the intent and grabbed the first fridge item. The LLM solves this at the only point where it matters:
**choosing which ingredient to search for**.

Everything else (checking the fridge, adding items, calling done) is still autopilot.
The LLM's one job is to read the natural language request and decide: *"comforting + cold evening
→ chicken curry or a warm stir fry → search for chicken"*.

That single reasoning step is what no amount of `if/else` can replicate.

In [ ]:
import os
import openai

os.environ['OPENAI_API_KEY'] = 'your-key-here'
try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except (ImportError, Exception):
    api_key = os.environ.get('OPENAI_API_KEY')

client = openai.OpenAI(
    api_key=api_key,
    base_url="https://litellm.sph-prod.ethz.ch/v1",
)
print('LLM client ready!')

LLM client ready!


In [8]:
# ── Layer 1: Autopilot — handle mechanical steps without the LLM ──────────

def _auto_handle_llm(chef, world, user_request):
    """Return an action string for obvious steps, or None to defer to LLM."""
    if not chef.fridge_contents:
        return 'ACTION: check_fridge()'

    if chef.chosen_recipe and chef.needed_ingredients:
        fridge_lower = [item.lower() for item in chef.fridge_contents]
        shopping_lower = [item.lower() for item in chef.shopping_list]
        all_covered = all(
            ing.lower() in fridge_lower or ing.lower() in shopping_lower
            for ing in chef.needed_ingredients
        )
        if all_covered:
            return 'ACTION: done()'

    return None


# ── Layer 2: Memory — structured summary for the LLM ─────────────────────

def _build_memory_llm(chef, world, history, user_request):
    """Build a structured context summary for the LLM."""
    sections = []

    sections.append(f'=== USER REQUEST ===\n  {user_request}')

    if chef.fridge_contents:
        items = ', '.join(chef.fridge_contents)
        sections.append(f'=== FRIDGE CONTENTS ===\n  {items}')

    if chef.possible_recipes:
        recipes = ', '.join(chef.possible_recipes)
        sections.append(f'=== POSSIBLE RECIPES ===\n  {recipes}')

    if chef.chosen_recipe:
        sections.append(f'=== CHOSEN RECIPE ===\n  {chef.chosen_recipe}')
        if chef.needed_ingredients:
            fridge_lower = [item.lower() for item in chef.fridge_contents]
            have = [i for i in chef.needed_ingredients if i.lower() in fridge_lower]
            missing = [i for i in chef.needed_ingredients if i.lower() not in fridge_lower]
            shopping_lower = [s.lower() for s in chef.shopping_list]
            still_need = [m for m in missing if m.lower() not in shopping_lower]
            sections.append(
                f'=== INGREDIENTS STATUS ===\n'
                f'  Have: {", ".join(have) if have else "(none)"}\n'
                f'  On shopping list: {", ".join(chef.shopping_list) if chef.shopping_list else "(empty)"}\n'
                f'  Still need to add: {", ".join(still_need) if still_need else "(all covered!)"}'
            )

    recent = [h for h in history[-8:] if h['role'] in ('action', 'result')]
    if recent:
        lines = [f'  {"Action" if h["role"] == "action" else "Result"}: {h["content"][:120]}'
                 for h in recent]
        sections.append('=== RECENT HISTORY ===\n' + '\n'.join(lines[-6:]))

    return '\n\n'.join(sections)


# ── Layer 3: System prompt ────────────────────────────────────────────────

SYSTEM_PROMPT_LLM = """You are a kitchen planning assistant helping someone decide what to cook.

GOAL: Help the user find a recipe that matches their request, using what's in the fridge.
Then build a shopping list for any missing ingredients.

AVAILABLE TOOLS:
{tools}

WORKFLOW:
1. Fridge is already checked — you know what's available.
2. INTERPRET the user's request to decide which ingredient to search for.
   - "something comforting for a cold evening" → think hearty, warming → search "chicken" or "rice"
   - "something quick after work" → think fast dishes → search "eggs" or "bread"
   - "I'm trying to eat lighter" → think fresh dishes → search "tomatoes" or "bell pepper"
   This interpretation is your most important job. A rule-based system cannot do this.
3. Search recipes using that ingredient.
4. Pick a recipe that fits the user's request.
5. Add missing ingredients ONE AT A TIME.
6. Call done() when the shopping list is complete.

CRITICAL RULES:
- Do NOT add items that are already in the fridge.
- Call done() only AFTER choosing a recipe and adding all missing ingredients.
- NEVER call check_fridge() — you already have the fridge contents above.

OUTPUT FORMAT — always show your reasoning, then one ACTION:

REASONING: "Comforting for a cold evening" suggests something warm and hearty like a stew or curry.
  I have chicken breast and rice — chicken curry would be perfect. Let me search for chicken.
ACTION: search_recipes(ingredient="chicken breast")

REASONING: Chicken curry fits perfectly and I already have chicken, rice, onions, and garlic.
  I just need coconut milk, curry powder, and ginger. Let me get the full ingredient list.
ACTION: get_ingredients(recipe="chicken curry")
"""


# ── Layer 4: The think function ───────────────────────────────────────────

def think_llm(chef: ChefState, world: KitchenWorld, history: list[dict], user_request: str) -> str:
    """LLM-powered kitchen agent — interprets natural language intent, autopilots the rest."""

    # ── Autopilot: mechanical steps ──
    auto = _auto_handle_llm(chef, world, user_request)
    if auto:
        return auto

    # ── LLM: interpret intent and make decisions ──
    memory = _build_memory_llm(chef, world, history, user_request)
    system = SYSTEM_PROMPT_LLM.format(tools=TOOLS_DESCRIPTION)

    user_msg = (
        f"{memory}\n\n"
        "What should you do next? Remember: your key job is interpreting the user's intent to "
        "pick the right ingredient to search for — that's what a rule-based agent cannot do."
    )

    try:
        response = client.chat.completions.create(
            model="anthropic/claude-sonnet-4-5",
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user_msg},
            ],
            max_tokens=400,
            temperature=0.3,
        )
        text = response.choices[0].message.content
    except Exception:
        # Fallback to rule-based logic on API error
        if not chef.possible_recipes and chef.fridge_contents:
            return f'ACTION: search_recipes(ingredient="{chef.fridge_contents[0]}")'
        if chef.possible_recipes and not chef.chosen_recipe:
            return f'ACTION: get_ingredients(recipe="{chef.possible_recipes[0]}")'
        if chef.chosen_recipe:
            return 'ACTION: done()'
        return 'ACTION: check_fridge()'

    # ── Guardrails ──
    tool_name, args = parse_action(text)

    # Block redundant fridge checks
    if tool_name == 'check_fridge' and chef.fridge_contents:
        if not chef.possible_recipes:
            return f'ACTION: search_recipes(ingredient="{chef.fridge_contents[0]}")'
        if chef.possible_recipes and not chef.chosen_recipe:
            return f'ACTION: get_ingredients(recipe="{chef.possible_recipes[0]}")'

    # Block adding items already in fridge
    if tool_name == 'add_to_shopping_list':
        item = args.get('item', '').lower()
        fridge_lower = [f.lower() for f in chef.fridge_contents]
        if item in fridge_lower:
            shopping_lower = [s.lower() for s in chef.shopping_list]
            for ing in chef.needed_ingredients:
                if ing.lower() not in fridge_lower and ing.lower() not in shopping_lower:
                    return f'ACTION: add_to_shopping_list(item="{ing}")'
            return 'ACTION: done()'

    return text

In [9]:
result_llm = play_with_llm(
    think_fn=lambda chef, world, history, req: think_llm(chef, world, history, req),
    user_request=USER_REQUEST,
    max_turns=20,
    delay=0.5,
)
print(f"\nRequest: '{USER_REQUEST}'")
print(f"Result:  {'Complete!' if result_llm['completed'] else 'Timed out'}")
print(f"Recipe:  {result_llm['recipe']}  ← chosen because it matches the intent")
print(f"Shopping: {', '.join(result_llm['shopping_list']) or 'nothing needed'}")
print(f"Turns:   {result_llm['turns']}")


Request: 'I want something comforting for a cold evening'
Result:  Complete!
Recipe:  chicken curry  ← chosen because it matches the intent
Shopping: coconut milk, curry powder, ginger
Turns:   8


In [10]:
# ── Side-by-side comparison ───────────────────────────────────────────────

print(f"Request: '{USER_REQUEST}'")
print()
print(f"{'':25} {'RULE-BASED':>20}    {'LLM':>20}")
print(f"{'─'*70}")
print(f"{'Ingredient searched':25} {result_rb.get('_ingredient', 'eggs (fallback)'):>20}    {'(interpreted from request)':>20}")
print(f"{'Recipe chosen':25} {result_rb['recipe']:>20}    {result_llm['recipe']:>20}")
print(f"{'Shopping list size':25} {len(result_rb['shopping_list']):>20}    {len(result_llm['shopping_list']):>20}")
print(f"{'Turns used':25} {result_rb['turns']:>20}    {result_llm['turns']:>20}")
print()
print("Rule-based: ignored the request, picked the first recipe alphabetically.")
print("LLM:        interpreted the intent, picked a recipe that actually fits.")

Request: 'I want something comforting for a cold evening'

                                    RULE-BASED                     LLM
──────────────────────────────────────────────────────────────────────
Ingredient searched            eggs (fallback)    (interpreted from request)
Recipe chosen                         omelette           chicken curry
Shopping list size                           2                       3
Turns used                                   7                       8

Rule-based: ignored the request, picked the first recipe alphabetically.
LLM:        interpreted the intent, picked a recipe that actually fits.


In [ ]:
# Download the game log - 
# NO THIS IS NOT THE WAY... TO GENERATE A MEANINGFUL DATASET ITS BETTER TO LET 2 AGENTS PLAY AGAINST EACH OTHER AND THEN DOWNLOAD THE LOGS
try:
    from google.colab import files
    files.download(result['log_file'])
except ImportError:
    print(f"Log file: {result['log_file']}")
    print('Open the file to inspect your agent\'s decisions turn by turn.')

---
## Phase 3: ML Classifier Think Function (Skeleton)

**Goal**: Replace the LLM with a trained classifier that predicts which tool to call.
This is the "distillation" step — we train a small model to mimic the LLM's behavior.

**Key steps**:
1. Generate synthetic training data from LLM game logs
2. Train a classifier to predict `tool_name` from state features
3. Use a heuristic to extract tool arguments (e.g., ingredient names)

This is a **skeleton** — the structure is complete but the model isn't trained.
Students fill in the training loop and feature engineering.

In [ ]:
import json
import glob
import numpy as np

# ═══════════════════════════════════════════════════════════════════════════
# STEP 1: Generate synthetic training data
# ═══════════════════════════════════════════════════════════════════════════
#
# Run the LLM agent multiple times with different free-form requests to build
# a dataset of (state, action) pairs.
#
# Note: the requests are intentionally vague — the LLM interprets them.
# This is what we want the classifier to eventually learn to replicate.

TRAINING_REQUESTS = [
    "I want something comforting for a cold evening",
    "Something quick I can throw together after work",
    "I'm trying to eat a bit lighter this week",
    "I want to impress someone for dinner tonight",
    "Something warm and filling for a rainy day",
    "I want to use up whatever needs to go first",
    "Something my kids will actually eat",
    "A proper hearty dinner, not a snack",
    "I feel like Asian food tonight",
    "Something simple, I don't want to spend long cooking",
]


def generate_training_data(requests=TRAINING_REQUESTS, max_turns=20):
    """Run the LLM agent on multiple requests and collect training data.

    TODO: Implement this function.

    Returns:
        list of dicts, each with:
            - 'features': dict of state features (see extract_features below)
            - 'label': str tool name that was chosen
            - 'args': dict of arguments passed to the tool
    """
    # training_data = []
    #
    # for request in requests:
    #     result = play_with_llm(
    #         think_fn=lambda c, w, h, r: think_llm(c, w, h, r),
    #         user_request=request,
    #         max_turns=max_turns,
    #         show_display=False,
    #     )
    #
    #     with open(result['log_file']) as f:
    #         log = json.load(f)
    #
    #     for turn_data in log['turns']:
    #         if 'action' not in turn_data:
    #             continue
    #         tool_name, args = parse_action(f"ACTION: {turn_data['action']}")
    #         features = {
    #             'has_fridge': 1 if turn_data.get('fridge_checked', False) else 0,
    #             'has_recipes': 1 if turn_data.get('possible_recipes') else 0,
    #             'has_chosen': 1 if turn_data.get('recipe') else 0,
    #             'shopping_count': len(turn_data.get('shopping_list', [])),
    #             'turn': turn_data['turn'],
    #         }
    #         training_data.append({
    #             'features': features,
    #             'label': tool_name,
    #             'args': args,
    #         })
    #
    # return training_data

    raise NotImplementedError("TODO: Run LLM agent to generate training data")


# ═══════════════════════════════════════════════════════════════════════════
# STEP 2: Feature extraction
# ═══════════════════════════════════════════════════════════════════════════
#
# Convert the chef state into a fixed-size feature vector for the classifier.
# The features should capture where we are in the pipeline.
#
# Note: the classifier only learns WHICH tool to call (check_fridge,
# search_recipes, get_ingredients, add_to_shopping_list, done).
# It cannot learn to interpret free-form requests — that's why argument
# extraction (step 4) still uses heuristics.

TOOL_NAMES = ['check_fridge', 'search_recipes', 'get_ingredients',
              'add_to_shopping_list', 'done']


def extract_features(chef: ChefState) -> np.ndarray:
    """Convert chef state into a feature vector for the classifier.

    TODO: Implement this function.

    Features to consider:
        - has_fridge_contents (0/1): Have we checked the fridge?
        - has_possible_recipes (0/1): Have we searched for recipes?
        - has_chosen_recipe (0/1): Have we picked a recipe?
        - num_needed (int): How many total ingredients does the recipe need?
        - num_have (int): How many ingredients do we already have?
        - num_missing (int): How many are still missing (not in fridge or shopping list)?
        - num_shopping (int): How many items are on the shopping list?

    Returns:
        np.ndarray of shape (7,) with the features above
    """
    # has_fridge = 1 if chef.fridge_contents else 0
    # has_recipes = 1 if chef.possible_recipes else 0
    # has_chosen = 1 if chef.chosen_recipe else 0
    # num_needed = len(chef.needed_ingredients)
    #
    # fridge_lower = [f.lower() for f in chef.fridge_contents]
    # shopping_lower = [s.lower() for s in chef.shopping_list]
    # num_have = sum(1 for i in chef.needed_ingredients if i.lower() in fridge_lower)
    # num_missing = sum(
    #     1 for i in chef.needed_ingredients
    #     if i.lower() not in fridge_lower and i.lower() not in shopping_lower
    # )
    # num_shopping = len(chef.shopping_list)
    #
    # return np.array([has_fridge, has_recipes, has_chosen,
    #                  num_needed, num_have, num_missing, num_shopping])

    raise NotImplementedError("TODO: Extract features from chef state")


# ═══════════════════════════════════════════════════════════════════════════
# STEP 3: Train the classifier
# ═══════════════════════════════════════════════════════════════════════════

def train_classifier(training_data):
    """Train a classifier on the collected training data.

    TODO: Implement this function.

    Args:
        training_data: list of dicts with 'features' and 'label' keys

    Returns:
        A trained sklearn model with a .predict() method
    """
    # from sklearn.tree import DecisionTreeClassifier
    #
    # X = np.array([list(d['features'].values()) for d in training_data])
    # y = np.array([TOOL_NAMES.index(d['label']) for d in training_data])
    #
    # model = DecisionTreeClassifier(max_depth=5)
    # model.fit(X, y)
    # print(f"Training accuracy: {model.score(X, y):.1%}")
    # return model

    raise NotImplementedError("TODO: Train a classifier")


# ═══════════════════════════════════════════════════════════════════════════
# STEP 4: Argument extraction heuristic
# ═══════════════════════════════════════════════════════════════════════════
#
# The classifier predicts WHICH tool to call — but not the arguments.
# For search_recipes we still fall back to the first fridge item
# (same limitation as the rule-based agent). That's fine: the classifier
# is only learning the pipeline structure, not intent interpretation.

def extract_args(tool_name: str, chef: ChefState, user_request: str) -> dict:
    """Determine tool arguments using heuristics.

    TODO: Implement this function.
    """
    # if tool_name == 'search_recipes':
    #     request_lower = user_request.lower()
    #     for item in chef.fridge_contents:
    #         if item.lower() in request_lower:
    #             return {'ingredient': item}
    #     return {'ingredient': chef.fridge_contents[0] if chef.fridge_contents else 'eggs'}
    #
    # if tool_name == 'get_ingredients':
    #     if chef.possible_recipes:
    #         return {'recipe': chef.possible_recipes[0]}
    #     return {'recipe': 'omelette'}
    #
    # if tool_name == 'add_to_shopping_list':
    #     fridge_lower = [f.lower() for f in chef.fridge_contents]
    #     shopping_lower = [s.lower() for s in chef.shopping_list]
    #     for ing in chef.needed_ingredients:
    #         if ing.lower() not in fridge_lower and ing.lower() not in shopping_lower:
    #             return {'item': ing}
    #     return {'item': ''}
    #
    # return {}

    raise NotImplementedError("TODO: Extract arguments for the predicted tool")


# ═══════════════════════════════════════════════════════════════════════════
# STEP 5: The think function using the trained classifier
# ═══════════════════════════════════════════════════════════════════════════

# model = None  # Set this after training: model = train_classifier(data)


def think_classifier(chef: ChefState, world: KitchenWorld, history: list[dict], user_request: str) -> str:
    """ML classifier think function — predicts tool from state features.

    TODO: Implement after completing steps 1-4.

    This function:
    1. Extracts features from the current chef state
    2. Uses the trained classifier to predict which tool to call
    3. Uses heuristics to determine the tool's arguments
    4. Returns the ACTION string
    """
    # features = extract_features(chef)
    # prediction = model.predict(features.reshape(1, -1))[0]
    # tool_name = TOOL_NAMES[prediction]
    # args = extract_args(tool_name, chef, user_request)
    # args_str = ', '.join(f'{k}="{v}"' for k, v in args.items())
    # return f'ACTION: {tool_name}({args_str})'

    raise NotImplementedError(
        "TODO: Complete steps 1-4, train the model, then implement this function"
    )

In [ ]:
# Once you've implemented all 5 steps above, uncomment and run:
#
# # Generate training data (requires GEMINI_API_KEY)
# data = generate_training_data()
# print(f"Collected {len(data)} training examples")
#
# # Train the classifier
# model = train_classifier(data)
#
# # Test it!
# result = play_rule_based(
#     think_fn=think_classifier,
#     user_request="I want to cook something with eggs",
#     max_turns=20,
#     delay=0.05,
# )
# print(f"\nResult: {'Complete!' if result['completed'] else 'Timed out'} | "
#       f"Recipe: {result['recipe']} | "
#       f"Shopping: {', '.join(result['shopping_list']) or 'nothing'} | "
#       f"Turns: {result['turns']}")